In [ ]:
%cd ../..
import os
import torch
import polars as pl
import pandas as pd
import pydicom
from tqdm import tqdm
from omegaconf import OmegaConf
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydicom")

from dinov2.inference import generate_embeddings, build_model, view_volume, crop_volume

In [ ]:
data_path = "/scratch/scratch1/RSNA-PE"

metadata_df = pl.read_csv(os.path.join(data_path, "train.csv"))
metadata_df

In [ ]:
study_uids = metadata_df["StudyInstanceUID"].unique().to_list()
series_uids = metadata_df["SeriesInstanceUID"].unique().to_list()
print("Study uids:", len(study_uids))
print("Series uids:", len(series_uids))


In [ ]:
def load_section(series_uid):
    series_df = metadata_df.filter(pl.col("SeriesInstanceUID") == series_uid)

    dcms = [(pydicom.dcmread(os.path.join(data_path, "train", row[0], row[1], f"{row[2]}.dcm")), row[3] == 1) for row in series_df.iter_rows()]
    dcms.sort(key=lambda x: x[0].ImagePositionPatient[2])

    slope = dcms[0][0].RescaleSlope
    inter = dcms[0][0].RescaleIntercept

    image_stack = torch.stack([torch.from_numpy(x[0].pixel_array) for x in dcms]) * slope + inter
    image_stack = image_stack.clip(-1000, 1900)
    labels = [x[1] for x in dcms]

    return image_stack, labels

image_stack, labels = load_section("57e3e3c5f910")
image_stack = crop_volume(image_stack)

view_volume(image_stack)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
)

In [ ]:
output_path = "/scratch/scratch1/embeddings/RSNA-PE/demo"
csv_output_path = os.path.join(output_path, "labels.csv")
os.makedirs(output_path, exist_ok=True)

new_labels_rows = []

if os.path.exists(csv_output_path):
    done_labels = pd.read_csv(os.path.join(output_path, "labels.csv"))
    done_series_uids = [sid for sid in series_uids if sid in done_labels["series_uid"].unique() and os.path.exists(os.path.join(output_path, f"{sid}.pth"))]

    done_labels = done_labels[done_labels["series_uid"].isin(done_series_uids)]
    todo_series_uids = [sid for sid in series_uids if sid not in done_series_uids]
else:
    todo_series_uids = series_uids

todo_series_uids = series_uids

for idx, series_uid in tqdm(enumerate(todo_series_uids), total=len(todo_series_uids)):
    img_output_dir = os.path.join(output_path, f"{series_uid}.pth")

    if os.path.exists(img_output_dir):
        continue
    
    img, labels = load_section(series_uid)
    img = crop_volume(img)
    D = img.shape[0]

    c: int = data_kwargs["channels"] # type: ignore
    d_mod2 = D % c
    d_mod = d_mod2 // 2

    img = img[d_mod : D - d_mod2 + d_mod, :, :]
    labels = labels[d_mod : D - d_mod2 + d_mod]

    labels = [labels[i:i+c] for i in range(0,len(labels), c)]
    num_positive = [sum(l) for l in labels]
    labels = [sum(l) > 0 for l in labels]
    
    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    for i in range(len(labels)):
        new_labels_rows.append((series_uid, i, labels[i], num_positive[i]))

    output = {"cls": collated_features["cls"]}

    torch.save(output, img_output_dir)


In [ ]:
columns = ["series_uid", "slice_idx", "has_pe", "num_img_pe"]

df = pd.DataFrame(new_labels_rows, columns=columns)
if os.path.exists(csv_output_path):
    df = pd.concat([done_labels, df], ignore_index=True)
df.to_csv(csv_output_path, index=False)